<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@700&display=swap" rel="stylesheet">

<h1 style="
font-family: 'Oswald', sans-serif;
font-weight: 700;
font-style: italic;
font-size: 90px;
letter-spacing: 2px;
color: #E7C173;
text-shadow: 3px 3px 0 #333;
">
MACHINE LEARNING<br>IN INDUSTRY
</h1>

# Day 2: Classical Models, Evaluation, and Diagnostics

This notebook covers the **core modeling pipeline**: split → preprocess → train → validate → evaluate → diagnose.
It picks up from Day 1's preprocessing work but includes a self-contained quick-load path.

## Table of Contents

- [0. Scope and Success Criteria](#scope)
- [1. Data Loading and Quick Recap](#data-loading)
- [2. Splitting Strategy](#splitting)
- [3. Evaluation Metrics](#metrics)
- [4. Model Zoo — 4 Families](#model-zoo)
  - [4.1 Logistic Regression](#lr)
  - [4.2 Decision Tree](#dt)
  - [4.3 Random Forest](#rf)
  - [4.4 Gradient Boosting](#gbm)
- [5. The Overfitting Problem](#overfitting)
- [6. Hyperparameter Tuning](#tuning)
- [7. Model Diagnostics](#diagnostics)
- [8. End-of-Notebook Checklist](#checklist)
- [Acceptance Checks](#acceptance)

## <a id="scope"></a> Section 0 — Scope and Success Criteria

**Scope for Day 2**
- Train and compare models across 4 classical families (linear, trees, ensembles, boosting).
- Understand splitting strategies and why stratified CV is the default.
- Evaluate models with threshold-dependent and threshold-independent metrics.
- Set up reproducible Pipelines with cross-validated hyperparameter tuning.
- Core diagnostics: permutation importance, calibration, model comparison.

**Success criteria**
- You can train and compare models from different families on the same preprocessed data.
- You can set up a reproducible Pipeline with cross-validated hyperparameter tuning.
- You can interpret diagnostic plots and identify overfitting.

**What's NOT in this notebook** (see `advanced_modeling.ipynb`):
SHAP, Optuna, adversarial validation, covariate shift, tabular foundation models, contrast model error analysis.

In [ ]:
import os
from pathlib import Path

# Walk up to the project root (idempotent, safe to re-run)
_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    KFold,
    GroupKFold,
    learning_curve,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve,
    classification_report,
    brier_score_loss,
    log_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingClassifier,
)
from tqdm import tqdm
SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

import xgboost as xgb
import lightgbm as lgb

def show(df, n=20):
    """Display helper."""
    display(df.head(n))

print("Environment ready.")
print(f"  xgboost:  {xgb.__version__}")
print(f"  lightgbm: {lgb.__version__}")

## <a id="data-loading"></a> Section 1 — Data Loading and Quick Recap

We reload the Adult Income dataset and re-apply the essential Day 1 preprocessing steps so this notebook is **self-contained**.

The steps below mirror what we did in Day 1:
1. **Load** the raw issues dataset and create the binary target.
2. **Dedup** — keep the latest record per `person_id` (sort by `record_written_at`, drop duplicates).
3. **Split** — use the dataset's `split` column for train/test, then carve out a validation set from train with stratified sampling.
4. **Classify columns** — separate numeric vs categorical features, exclude leakage and process columns.
5. **Build preprocessing pipelines** — one with scaling (for linear models) and one without (for tree-based models).

The goal is to arrive at clean `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test` matrices ready for modeling.

In [ ]:
# ── Load raw data ──
adult = pd.read_csv("day1/generated/adult_income_issues.csv")

TARGET_COL = "class"
TARGET_BIN_COL = "target"
SPLIT_COL = "split"
ID_COLS = ["person_id"]

LEAKAGE_COLS = ["post_adjudication_risk_code"]

PROCESS_COLS = [
    "db_source_table", "db_etl_batch_id", "db_row_surrogate_key",
    "db_loaded_at_utc", "dataset_schema_version", "extract_country_code",
    "record_written_at", "dgp_regime",
]

# Binary target
adult[TARGET_BIN_COL] = (
    adult[TARGET_COL].astype(str).str.contains(">50", case=False, regex=False).astype(int)
)

# Dedup: keep latest record per person (same logic as Day 1)
adult = (adult.sort_values("record_written_at", ascending=False)
         .drop_duplicates("person_id", keep="first")
         .reset_index(drop=True))

print("Adult shape (after dedup):", adult.shape)
print("Target positive rate:", round(adult[TARGET_BIN_COL].mean(), 4))

In [ ]:
# ── Split: use the dataset's split column, then carve out validation from train ──
def to_numeric_loose(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip().str.replace("h", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce")

train_pool = adult.loc[adult[SPLIT_COL] == "train"].copy()
test_df = adult.loc[adult[SPLIT_COL] == "test"].copy()

# After dedup, each person_id has exactly one row → simple stratified split
train_df, val_df = train_test_split(
    train_pool, test_size=0.2, random_state=SEED,
    stratify=train_pool[TARGET_BIN_COL],
)

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5} rows={len(frame):5d}  target_rate={frame[TARGET_BIN_COL].mean():.4f}")

print("Split integrity check passed.")

In [ ]:
# ── Quick preprocessing (compact Day 1 recap) ──
# Drop non-feature columns
excluded = set(
    [TARGET_COL, TARGET_BIN_COL, SPLIT_COL]
    + ID_COLS + LEAKAGE_COLS + PROCESS_COLS
    + ["case_review_note", "constant_one"]
)
feature_cols = [c for c in adult.columns if c not in excluded]

# Classify into numeric vs categorical
numeric_cols = []
categorical_cols = []
for col in feature_cols:
    s = train_df[col]
    if s.dtype.kind in "biufc":
        numeric_cols.append(col)
        continue
    as_num = to_numeric_loose(s)
    if as_num.notna().mean() >= 0.9:
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# ── Build feature matrices ──
def build_features(df, numeric_cols, categorical_cols):
    X = pd.DataFrame(index=df.index)
    for c in numeric_cols:
        X[c] = to_numeric_loose(df[c])
    for c in categorical_cols:
        X[c] = df[c].astype(str).fillna("MISSING")
    return X

X_train_raw = build_features(train_df, numeric_cols, categorical_cols)
X_val_raw = build_features(val_df, numeric_cols, categorical_cols)
X_test_raw = build_features(test_df, numeric_cols, categorical_cols)

y_train = train_df[TARGET_BIN_COL].values
y_val = val_df[TARGET_BIN_COL].values
y_test = test_df[TARGET_BIN_COL].values

# ── Build preprocessing pipelines ──
numeric_transformer_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
numeric_transformer_simple = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# For scaling-sensitive models (LogReg)
preprocessor_scaled = ColumnTransformer([
    ("num", numeric_transformer_scaled, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])
# For tree-based models (DT, RF, GBM)
preprocessor_tree = ColumnTransformer([
    ("num", numeric_transformer_simple, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

# Fit on train, transform all sets
preprocessor_scaled.fit(X_train_raw)
X_train_sc = preprocessor_scaled.transform(X_train_raw)
X_val_sc = preprocessor_scaled.transform(X_val_raw)
X_test_sc = preprocessor_scaled.transform(X_test_raw)

preprocessor_tree.fit(X_train_raw)
X_train_tr = preprocessor_tree.transform(X_train_raw)
X_val_tr = preprocessor_tree.transform(X_val_raw)
X_test_tr = preprocessor_tree.transform(X_test_raw)

feature_names_ohe = preprocessor_tree.get_feature_names_out()

print(f"\nFeature matrix shapes for Linear Models: train={X_train_sc.shape}, val={X_val_sc.shape}, test={X_test_sc.shape}")
print(f"\nFeature matrix shapes for Tree-Based Models: train={X_train_tr.shape}, val={X_val_tr.shape}, test={X_test_tr.shape}")
print(f"No NaN in train (scaled):  {np.isfinite(X_train_sc).all()}")
print(f"No NaN in train (tree):    {np.isfinite(X_train_tr).all()}")

## <a id="splitting"></a> Section 2 — Splitting Strategy

### Why split at all?

When you train a model, it adjusts its parameters to **minimize error on the training data**.
If you then evaluate it on *the same data*, the model has already "seen the answers" — the error estimate is **optimistically biased**.

This is not a subtle effect. A sufficiently flexible model (e.g., an unpruned decision tree) can achieve **perfect** training accuracy on almost any dataset — while being completely useless on new data.

The solution: **hold out data the model never sees during training**, and evaluate on that.

📚 James et al., [*An Introduction to Statistical Learning*](https://www.statlearning.com/), Chapter 5 — the gold-standard introduction to resampling methods (free PDF).

In [ ]:
# ── Why split? Train-set evaluation is misleading ──
from sklearn.tree import DecisionTreeClassifier

dt_overfit = DecisionTreeClassifier(max_depth=None, random_state=SEED)
dt_overfit.fit(X_train_tr, y_train)

train_auc = roc_auc_score(y_train, dt_overfit.predict_proba(X_train_tr)[:, 1])
val_auc = roc_auc_score(y_val, dt_overfit.predict_proba(X_val_tr)[:, 1])

print("Unpruned Decision Tree:")
print(f"  Train AUC: {train_auc:.4f}  ← the model memorized the training data")
print(f"  Val AUC:   {val_auc:.4f}  ← actual performance on unseen data")
print(f"  Gap:       {train_auc - val_auc:.4f}")
print()
print("If you only looked at the training score, you'd think this model is perfect.")
print("That's why you ALWAYS need held-out data for evaluation.")

### Why cross-validation instead of a single holdout?

OK, so we need held-out data. The simplest approach is a **single train/test split** (holdout).
The problem: that split is **one random draw**. A different random seed gives different rows in the test set, and a different AUC estimate.

With small or moderate datasets, this variance can be **surprisingly large** — your reported AUC might be 0.82 or 0.88 depending on which rows happened to land in the test set.

**Cross-validation** averages over K different splits, giving you:
1. A **more stable estimate** of performance (the mean).
2. A **measure of uncertainty** (the standard deviation across folds).

The demo below runs the same model on 50 different random holdout splits and compares the spread to a single 5-fold CV estimate.

In [ ]:
# ── Holdout variance vs cross-validation stability ──
from sklearn.ensemble import GradientBoostingClassifier

model_for_demo = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=SEED,
)

# --- 50 random holdout splits ---
holdout_aucs = []
for seed_i in tqdm(range(25)):
    X_ho_tr, X_ho_te, y_ho_tr, y_ho_te = train_test_split(
        X_train_tr, y_train, test_size=0.2, random_state=seed_i,
    )
    model_for_demo.fit(X_ho_tr, y_ho_tr)
    auc_i = roc_auc_score(y_ho_te, model_for_demo.predict_proba(X_ho_te)[:, 1])
    holdout_aucs.append(auc_i)

holdout_aucs = np.array(holdout_aucs)

# --- 5-fold CV (single run, but uses all data) ---
cv_scores_demo = cross_val_score(
    GradientBoostingClassifier(n_estimators=50, max_depth=5, learning_rate=0.1, random_state=SEED),
    X_train_tr, y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring="roc_auc",
)

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(holdout_aucs, bins=15, alpha=0.6, color="steelblue", edgecolor="white",
        label=f"25 holdout splits (std={holdout_aucs.std():.4f})")
ax.axvline(holdout_aucs.mean(), color="steelblue", linestyle="--", lw=2,
           label=f"Holdout mean = {holdout_aucs.mean():.4f}")
ax.axvline(cv_scores_demo.mean(), color="crimson", linestyle="-", lw=2,
           label=f"5-fold CV mean = {cv_scores_demo.mean():.4f} ± {cv_scores_demo.std():.4f}")
ax.axvspan(cv_scores_demo.mean() - cv_scores_demo.std(),
           cv_scores_demo.mean() + cv_scores_demo.std(),
           alpha=0.2, color="crimson", label="CV ± 1 std")
ax.set_xlabel("AUC")
ax.set_ylabel("Count")
ax.set_title("Holdout Variance vs Cross-Validation Stability")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Holdout range: {holdout_aucs.min():.4f} – {holdout_aucs.max():.4f}  (spread = {holdout_aucs.max() - holdout_aucs.min():.4f})")
print(f"CV estimate:   {cv_scores_demo.mean():.4f} ± {cv_scores_demo.std():.4f}")
print()
print("A single holdout split could land anywhere in that histogram.")
print("CV gives you a stable estimate + uncertainty — much more trustworthy for model selection.")

### What about Leave-One-Out (LOOCV)?

If K-fold is good, why not set K = N (one sample per fold)? This is **Leave-One-Out Cross-Validation**.

In theory it uses the maximum amount of training data per fold. In practice:
- **Computational cost**: you train N separate models — impractical for large datasets.
- **High variance**: each training set differs by only one sample, so the N estimates are highly correlated. The average can be *less* stable than 5- or 10-fold CV.

LOOCV is sometimes useful for very small datasets (N < 100), but **5- or 10-fold CV is the standard default** in practice.

📚 Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, §7.10 — discusses the bias-variance tradeoff of different K values.

### Which CV strategy? It depends on data structure

Now that we know *why* to cross-validate, the next question is *how* to split the folds.
The right strategy depends on the **structure of your data**:
- **No structure** → `KFold` (baseline) or `StratifiedKFold` (classification default)
- **Group structure** (e.g., multiple rows per patient/store) → `GroupKFold`
- **Time structure** (e.g., transactions over months) → `TimeSeriesSplit`

We demonstrate all four below with visualizations.

### Splitting Strategies at a Glance

| Strategy | When to use | Key idea |
|---|---|---|
| **KFold** | Generic baseline, no structure in data | Randomly partitions into K folds |
| **StratifiedKFold** | Classification (especially imbalanced) | Preserves target distribution in each fold |
| **GroupKFold** | Repeated measurements per entity (patient, store, user) | Entire group in one fold — prevents entity leakage |
| **TimeSeriesSplit** | Temporally ordered data | Train on past, validate on future — respects the arrow of time |

Below we visualize how each strategy assigns samples to train/test across folds.

📚 [sklearn Cross-validation Guide](https://scikit-learn.org/stable/modules/cross_validation.html) · [sklearn CV Visualization](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html)

In [ ]:
# ── Visual gallery: how each CV strategy assigns train/test ──
from matplotlib.patches import Patch
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit

cmap_cv = plt.cm.coolwarm


def plot_cv_indices(cv, X, y, ax, groups=None, title=None):
    """Visualize train/test assignments for each fold of a CV splitter."""
    n_splits = cv.get_n_splits(X, y, groups)
    for ii, (tr, tt) in enumerate(cv.split(X=X, y=y, groups=groups)):
        indices = np.array([np.nan] * len(X))
        indices[tt] = 1
        indices[tr] = 0
        ax.scatter(range(len(indices)), [ii + 0.5] * len(indices),
                   c=indices, marker="_", lw=7, cmap=cmap_cv,
                   vmin=-0.2, vmax=1.2)
    ax.set(yticks=np.arange(n_splits) + 0.5,
           yticklabels=range(n_splits),
           xlabel="Sample index", ylabel="Fold",
           ylim=[n_splits + 0.2, -0.2])
    ax.set_title(title or type(cv).__name__, fontsize=12, fontweight="bold")


# ── Toy dataset: 100 samples, 3 imbalanced classes, 5 groups ──
n_pts, n_splits = 100, 5
rng_cv = np.random.RandomState(SEED)
X_toy = rng_cv.randn(n_pts, 10)
y_toy = np.hstack([[ii] * int(n_pts * p) for ii, p in enumerate([0.1, 0.3, 0.6])])
groups_toy = np.repeat(np.arange(5), 20)  # 5 groups of 20

cvs = [
    (KFold(n_splits, shuffle=True, random_state=SEED), None),
    (StratifiedKFold(n_splits, shuffle=True, random_state=SEED), None),
    (GroupKFold(n_splits), groups_toy),
    (TimeSeriesSplit(n_splits), None),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
for ax, (cv, grp) in zip(axes.ravel(), cvs):
    plot_cv_indices(cv, X_toy, y_toy, ax, groups=grp)

# Shared legend
axes[0, 0].legend(
    [Patch(color=cmap_cv(0.02)), Patch(color=cmap_cv(0.8))],
    ["Training set", "Test set"], loc="upper right", fontsize=8,
)
fig.suptitle("Cross-Validation Strategies — Fold Assignments on 100 Samples",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("KFold:           random partition — no structure preserved.")
print("StratifiedKFold:  preserves class proportions in each fold.")
print("GroupKFold:       entire group stays together (no entity leakage).")
print("TimeSeriesSplit:  always trains on past, tests on future — growing window.")

In [ ]:
# ── Stratified split: target rate preservation ──
X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(
    X_train_tr, y_train, test_size=0.2, random_state=SEED,
)
X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
    X_train_tr, y_train, test_size=0.2, random_state=SEED, stratify=y_train,
)

print("Random split target rates:     "
      f"train={y_r_train.mean():.4f}, test={y_r_test.mean():.4f}")
print("Stratified split target rates:  "
      f"train={y_s_train.mean():.4f}, test={y_s_test.mean():.4f}")
print(f"Full train target rate:         {y_train.mean():.4f}")
print("\nStratified split preserves the target distribution — critical for imbalanced problems.")

In [ ]:
# ── Cross-validation in 5 lines ──
cv_model = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(cv_model, X_train_sc, y_train, cv=skf, scoring="roc_auc")

print(f"StratifiedKFold (5 folds): AUC = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Individual folds: {[f'{s:.4f}' for s in cv_scores]}")
print("\nCV gives you an estimate of generalization performance + its uncertainty.")

### When the wrong split misleads you

The visual gallery shows *how* each strategy splits data. But **why does it matter?**
Using the wrong strategy doesn't just give you a slightly different number — it can give you a
**fundamentally misleading estimate** of how your model will perform in production.

We show two concrete examples:
1. **Time-based data with a random split** — the model "sees the future" and you overestimate performance.
2. **Grouped data with KFold** — the model memorizes patient-level patterns and you overestimate generalization.

#### Example 1: Loan defaults with temporal drift

Imagine a bank building a loan default model. The economy shifts over time — default rates rise during downturns.
If you randomly shuffle loans from 2018–2023 into train/test, the model gets to "peek into the future":
it trains on 2023 loans (recession) and tests on 2020 loans (boom), learning patterns that wouldn't be available at prediction time.

The synthetic dataset below simulates this: default rates drift upward over time, and a feature correlated with the economic regime is available. A random split leaks future economic conditions into training.

In [ ]:
# ── Synthetic loan data with temporal drift ──
rng_loan = np.random.RandomState(42)
n_loans = 6000
months = np.arange(n_loans)  # proxy for time (loan origination order)

# Economic regime drifts: default-friendly features shift over time
economic_stress = months / n_loans  # 0→1 over time, simulates worsening economy

# Features: credit score, debt-to-income, loan amount + a time-correlated macro feature
credit_score = rng_loan.normal(700, 50, n_loans) - economic_stress * 30  # scores drift down
dti = rng_loan.uniform(0.1, 0.6, n_loans) + economic_stress * 0.15       # DTI drifts up
loan_amount = rng_loan.lognormal(10, 0.5, n_loans)
unemployment_rate = 4.0 + economic_stress * 6 + rng_loan.normal(0, 0.5, n_loans)  # macro feature

# Default probability depends on features + time-varying economic stress
logit = (-2.8 + economic_stress * 1.8
         + 0.5 * (dti - 0.3)
         + 0.08 * (unemployment_rate - 5)
         - 0.004 * (credit_score - 680)
         + rng_loan.normal(0, 0.8, n_loans))
default_prob = 1 / (1 + np.exp(-logit))
default = rng_loan.binomial(1, default_prob)

loan_df = pd.DataFrame({
    "credit_score": credit_score, "dti": dti,
    "loan_amount": loan_amount, "unemployment_rate": unemployment_rate,
    "month_idx": months, "default": default,
})

print(f"Loan dataset: {len(loan_df)} loans, {default.mean():.1%} overall default rate")
print(f"Default rate — first half: {default[:3000].mean():.1%}, second half: {default[3000:].mean():.1%}")
print("There is clear temporal drift in default rates (economy worsens over time).")

# --- Compare: random split vs time-based split ---
from sklearn.model_selection import TimeSeriesSplit

features_loan = ["credit_score", "dti", "loan_amount", "unemployment_rate"]
X_loan = loan_df[features_loan].values
y_loan = loan_df["default"].values

gbm_loan = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED)

# Random CV (WRONG for time-ordered data)
random_cv_scores = cross_val_score(
    gbm_loan, X_loan, y_loan,
    cv=KFold(5, shuffle=True, random_state=SEED), scoring="roc_auc",
)

# Time-based CV (CORRECT)
time_cv_scores = cross_val_score(
    gbm_loan, X_loan, y_loan,
    cv=TimeSeriesSplit(5), scoring="roc_auc",
)

print(f"\nRandom KFold CV:      AUC = {random_cv_scores.mean():.4f} ± {random_cv_scores.std():.4f}  ← optimistic!")
print(f"TimeSeriesSplit CV:   AUC = {time_cv_scores.mean():.4f} ± {time_cv_scores.std():.4f}  ← realistic")
print(f"Overestimation:       {random_cv_scores.mean() - time_cv_scores.mean():.4f} AUC points")
print()
print("The random split lets the model train on future economic conditions and test on past ones.")
print("In production, you only have the past — TimeSeriesSplit reflects this constraint.")

#### Example 2: Patient data with grouped observations

A classic Kaggle pitfall. Imagine a medical dataset where each **patient** has multiple visits.
Measurements from the same patient are correlated (same genetics, same lifestyle, same baseline health).

If you use regular KFold, the same patient can appear in both train and test — the model doesn't need to
*generalize to new patients*, it just needs to *recognize patients it already knows*. The CV score is inflated.

`GroupKFold` ensures all visits from a patient stay in the same fold. The score drops — but it reflects
what happens when you deploy the model on **patients it has never seen**.

This exact issue has appeared in major Kaggle competitions (e.g., [RSNA medical imaging](https://www.kaggle.com/c/rsna-intracranial-hemorrhage-detection), [Instant Gratification](https://www.kaggle.com/c/instant-gratification)) where top solutions emphasized correct grouping as critical to a reliable CV.

In [ ]:
# ── Synthetic patient data: multiple visits per patient ──
rng_med = np.random.RandomState(42)
n_patients = 200
visits_per_patient = 10  # each patient has 10 visits

patient_ids = np.repeat(np.arange(n_patients), visits_per_patient)
n_total = len(patient_ids)

# Each patient has a latent "health profile" — this is what makes visits correlated
patient_baseline = rng_med.normal(0, 1.5, n_patients)  # strong patient effect
patient_risk = np.repeat(patient_baseline, visits_per_patient)

# Visit-level features (noisy measurements around the patient's baseline)
blood_pressure = 120 + patient_risk * 10 + rng_med.normal(0, 5, n_total)
cholesterol = 200 + patient_risk * 15 + rng_med.normal(0, 10, n_total)
bmi = 25 + patient_risk * 3 + rng_med.normal(0, 2, n_total)
glucose = 90 + patient_risk * 8 + rng_med.normal(0, 8, n_total)

# Outcome: disease risk driven largely by patient-level effect
logit_med = 0.8 * patient_risk + rng_med.normal(0, 0.5, n_total)
disease = (logit_med > 0.5).astype(int)

X_med = np.column_stack([blood_pressure, cholesterol, bmi, glucose])
y_med = disease
groups_med = patient_ids

print(f"Medical dataset: {n_patients} patients × {visits_per_patient} visits = {n_total} rows")
print(f"Disease prevalence: {y_med.mean():.1%}")

# --- Compare: KFold (leaky) vs GroupKFold (correct) ---
from sklearn.ensemble import RandomForestClassifier

rf_med = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED)

# KFold — same patient can appear in train AND test (leaky)
kfold_scores = cross_val_score(
    rf_med, X_med, y_med,
    cv=KFold(5, shuffle=True, random_state=SEED), scoring="roc_auc",
)

# GroupKFold — entire patient stays in one fold (correct)
gkf_scores = cross_val_score(
    rf_med, X_med, y_med,
    cv=GroupKFold(5), scoring="roc_auc", groups=groups_med,
)

print(f"\nKFold CV (leaky):     AUC = {kfold_scores.mean():.4f} ± {kfold_scores.std():.4f}  ← inflated!")
print(f"GroupKFold (correct): AUC = {gkf_scores.mean():.4f} ± {gkf_scores.std():.4f}  ← realistic")
print(f"Overestimation:       {kfold_scores.mean() - gkf_scores.mean():.4f} AUC points")
print()
print("With KFold, the model sees 8 visits from a patient in train and predicts the remaining 2.")
print("It doesn't learn medicine — it learns to recognize patients. GroupKFold forces real generalization.")

#### Takeaway

**The split strategy is not a technicality — it's part of the problem definition.**
Before writing any modeling code, ask: *"what will my model see at prediction time?"* and make sure your CV mirrors that constraint.

| Data structure | Wrong split | Right split | What goes wrong |
|---|---|---|---|
| Time-ordered | Random KFold | TimeSeriesSplit | Model trains on the future, tests on the past |
| Grouped entities | Regular KFold | GroupKFold | Model memorizes entities instead of learning patterns |
| Imbalanced classes | Unstratified KFold | StratifiedKFold | Some folds may have 0 positives |

## <a id="metrics"></a> Section 3 — Evaluation Metrics for Binary Classification

Before training any models, you need to know **what you're measuring**.
Metrics for binary classification fall into three classes:

| Class | What it measures | Examples |
|---|---|---|
| **Threshold-dependent** | Quality of *hard decisions* (yes/no) | Accuracy, precision, recall, F1 |
| **Ranking** | Quality of the *ordering* | ROC-AUC, AUPRC, lift@K |
| **Probabilistic** | Quality of predicted *probabilities* | Log loss, Brier score |

We'll define each class with formulas and code, then discuss when to prefer one over another.

### 3.1 — Threshold-Dependent Metrics

Every threshold-dependent metric starts from the **confusion matrix** — a 2×2 table of correct and incorrect predictions:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actually Positive** | True Positive (TP) | False Negative (FN) |
| **Actually Negative** | False Positive (FP) | True Negative (TN) |

From this table:

- **Accuracy** = (TP + TN) / (TP + TN + FP + FN) — fraction of all predictions that are correct
- **Precision** = TP / (TP + FP) — "of everything I flagged as positive, how many actually are?"
- **Recall** (sensitivity) = TP / (TP + FN) — "of all actual positives, how many did I catch?"
- **F1** = 2 × (Precision × Recall) / (Precision + Recall) — harmonic mean, balances both

All of these **depend on the threshold** you choose: lowering the threshold catches more positives (recall up) but also flags more false positives (precision down).

In [ ]:
# ── Quick demo: confusion matrix from a dummy model ──
# We'll use a simple threshold on random probabilities to illustrate the metrics
rng = np.random.RandomState(SEED)
y_demo = rng.binomial(1, 0.3, size=1000)
y_prob_demo = y_demo * rng.uniform(0.4, 0.9, size=1000) + (1 - y_demo) * rng.uniform(0.1, 0.6, size=1000)

threshold = 0.5
y_pred_demo = (y_prob_demo >= threshold).astype(int)

cm = confusion_matrix(y_demo, y_pred_demo)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion matrix (threshold={threshold}):")
print(f"  TP={tp}  FN={fn}")
print(f"  FP={fp}  TN={tn}")
print()
print(f"Accuracy:  {accuracy_score(y_demo, y_pred_demo):.4f}")
print(f"Precision: {precision_score(y_demo, y_pred_demo):.4f}  ← of flagged positives, how many are real?")
print(f"Recall:    {recall_score(y_demo, y_pred_demo):.4f}  ← of actual positives, how many caught?")
print(f"F1:        {f1_score(y_demo, y_pred_demo):.4f}")

##### The Accuracy Trap

Accuracy **looks good by default on imbalanced data** — even a model that predicts "negative" for everyone achieves high accuracy if most data is negative.

In [ ]:
# ── The accuracy trap: a model that always predicts 0 ──
# Simulate a 5% positive-rate problem
rng = np.random.RandomState(SEED)
y_imb = rng.binomial(1, 0.05, size=10_000)
y_all_neg = np.zeros_like(y_imb)

print(f"Imbalanced dataset: {len(y_imb):,} rows, {y_imb.mean():.1%} positive")
print()
print(f"Always-predict-negative baseline:")
print(f"  Accuracy:  {accuracy_score(y_imb, y_all_neg):.4f}  ← looks great!")
print(f"  Precision: {precision_score(y_imb, y_all_neg, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_imb, y_all_neg):.4f}  ← catches zero positives")
print(f"  F1:        {f1_score(y_imb, y_all_neg):.4f}")
print()
print("95% accuracy, 0% recall. The model is useless but accuracy says it's great.")
print("Never use accuracy as the primary metric for imbalanced problems.")

### 3.2 — Ranking Metrics

Ranking metrics don't require choosing a threshold — they evaluate whether the model **orders** observations correctly.

- **ROC-AUC**: probability that a randomly chosen positive is scored higher than a randomly chosen negative.
  Ranges from 0.5 (random) to 1.0 (perfect). It is **prevalence-independent** — stable across datasets, but can be
  misleadingly optimistic when positives are rare.

- **PR-AUC** (Average Precision): area under the precision-recall curve. Because precision depends on the number
  of false positives, PR-AUC is **more informative when the positive class is rare**. A model with ROC-AUC = 0.90
  might have PR-AUC = 0.20 on a 1% positive-rate dataset.

- **Lift@K**: "if I act on the top K predictions, how many times better am I than random?" The metric of choice
  when you have a **fixed action budget** (e.g., 200 outbound calls per month).

📚 [Saito & Rehmsmeier (2015), "The Precision-Recall Plot Is More Informative than the ROC Plot"](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0118432) · [Meaningful Metrics: Cumulative Gains and Lift Charts](https://towardsdatascience.com/meaningful-metrics-cumulative-gains-and-lyft-charts-7aac02fc5c14)

In [ ]:
# ── ROC-AUC and PR-AUC on the dummy example ──
roc = roc_auc_score(y_demo, y_prob_demo)
pr = average_precision_score(y_demo, y_prob_demo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_demo, y_prob_demo)
axes[0].plot(fpr, tpr, lw=2, label=f"Model (AUC={roc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC=0.5)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_demo, y_prob_demo)
baseline_rate = y_demo.mean()
axes[1].plot(rec, prec, lw=2, label=f"Model (AP={pr:.3f})")
axes[1].axhline(baseline_rate, color="k", linestyle="--", alpha=0.4, label=f"Random ({baseline_rate:.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"ROC-AUC: {roc:.4f} — how well does the model rank positives above negatives?")
print(f"PR-AUC:  {pr:.4f} — how precise is the model across all recall levels?")

### 3.3 — Probabilistic Metrics (Reference)

Two models can have identical AUC but very different probability quality. If you use probabilities for **downstream calculations** (expected loss, pricing, risk scoring), you need **calibrated** outputs.

- **Log loss** (cross-entropy): penalizes confident wrong predictions heavily.
- **Brier score**: mean squared error of probabilities. 0 = perfect, 0.25 = random.

We'll revisit calibration briefly in §7. The **Advanced** notebook has a full demo showing two models with identical AUC but very different calibration.

### 3.4 — Choosing the Right Metric: It Comes From the Business

Now that you know what each metric measures, the question is: **which one should you optimize?**
The answer is never purely technical — it comes from the cost structure of the problem.

| Scenario | What matters | Metric to focus on |
|---|---|---|
| **Churn prediction** — you can call 200 customers/month | Of the top 200 flagged, how many are real churners? | Precision@K, lift@K |
| **Credit default** — missing a default loses the loan amount | Catch as many defaults as possible, tolerate false alarms | Recall |
| **Spam filter** — a false positive (real email in spam) is worse than a missed spam | Be very sure before deleting an email | Precision |
| **Loss Given Default** — predicted probability feeds into expected-loss pricing | The probability itself must be accurate, not just the ranking | Brier score, log loss |
| **Fraud detection** — analysts review flagged transactions, limited capacity | Of flagged cases, how many are real fraud? | Precision |
| **Medical screening** — missing a disease is much worse than an extra test | Catch every positive, even at cost of false positives | Recall at acceptable precision |

**Rule of thumb:** if you can't explain *why* you chose a metric in one sentence that references the business cost, you probably haven't thought about it enough.

### Beyond Binary Classification

This notebook focuses on binary classification metrics, but the same thinking applies elsewhere:

- **Multiclass**: accuracy, macro/micro/weighted F1, Cohen's kappa, confusion matrix per class. See [sklearn multiclass metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#multiclass-and-multilabel-classification).
- **Regression**: MAE, RMSE, R², MAPE. See [sklearn regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).
- **Forecasting / time series**: MASE, sMAPE, coverage of prediction intervals. See [Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*](https://otexts.com/fpp3/accuracy.html).

The principle is the same: **define what "good" means for your problem before you train anything.**

📚 [sklearn Model Evaluation Guide](https://scikit-learn.org/stable/modules/model_evaluation.html)

## <a id="model-zoo"></a> Section 4 — Model Zoo: 4 Families

We train one model per family on the same data, collect metrics in a registry, and compare.
For each model: brief intro, key hyperparameters, preprocessing needs, fit + evaluate.

In [ ]:
# ── Helper: evaluate a model and store results ──
MODEL_REGISTRY = {}
RESULTS_ROWS = []


def count_parameters(model):
    """Estimate the number of learned parameters for common sklearn models."""
    if hasattr(model, "coef_"):
        return int(np.prod(model.coef_.shape)) + int(np.prod(model.intercept_.shape))
    if hasattr(model, "tree_"):
        return int(model.tree_.node_count)
    if hasattr(model, "estimators_"):
        total = 0
        for est in (model.estimators_ if hasattr(model.estimators_[0], "tree_") else []):
            e = est if hasattr(est, "tree_") else est[0]
            total += e.tree_.node_count
        return total
    return None


def evaluate_model(name, model, X_tr, y_tr, X_v, y_v, fit=True):
    """Fit (optionally), predict, compute metrics, store in registry."""
    t0 = time.time()
    if fit:
        model.fit(X_tr, y_tr)
    fit_time = time.time() - t0

    y_pred_tr = model.predict(X_tr)
    y_pred_v = model.predict(X_v)

    if hasattr(model, "predict_proba"):
        y_prob_tr = model.predict_proba(X_tr)[:, 1]
        y_prob_v = model.predict_proba(X_v)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob_tr = model.decision_function(X_tr)
        y_prob_v = model.decision_function(X_v)
    else:
        y_prob_tr = y_pred_tr.astype(float)
        y_prob_v = y_pred_v.astype(float)

    row = {
        "model": name,
        "train_auc": roc_auc_score(y_tr, y_prob_tr),
        "val_auc": roc_auc_score(y_v, y_prob_v),
        "val_f1": f1_score(y_v, y_pred_v),
        "val_precision": precision_score(y_v, y_pred_v, zero_division=0),
        "val_recall": recall_score(y_v, y_pred_v),
        "n_params": count_parameters(model),
        "fit_time_s": fit_time,
    }
    RESULTS_ROWS.append(row)
    MODEL_REGISTRY[name] = {
        "model": model,
        "y_pred_val": y_pred_v,
        "y_prob_val": y_prob_v,
        "y_pred_train": y_pred_tr,
        "y_prob_train": y_prob_tr,
    }

    print(f"[{name}]")
    print(f"  Train AUC: {row['train_auc']:.4f}  |  Val AUC: {row['val_auc']:.4f}")
    print(f"  Val F1: {row['val_f1']:.4f}  |  Precision: {row['val_precision']:.4f}  |  Recall: {row['val_recall']:.4f}")
    print(f"  Parameters: {row['n_params']}  |  Fit time: {fit_time:.2f}s")
    return model

### <a id="lr"></a> 4.1 — Logistic Regression

A linear model that estimates the log-odds of the positive class as a linear combination of features.
Simple, fast, and highly interpretable through its coefficients.

**Key hyperparameters:** `C` controls regularization strength (smaller = stronger). `penalty` selects L1 (sparse) or L2 (ridge).
**Needs scaling:** Yes — features must be on the same scale for regularization to work correctly.

📚 [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) · [sklearn Linear Models Guide](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)

In [ ]:
lr = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000, random_state=SEED)
evaluate_model("LogisticRegression", lr, X_train_sc, y_train, X_val_sc, y_val)

# Inspect coefficients (top features by absolute weight)
coef_df = pd.DataFrame({
    "feature": feature_names_ohe,
    "coef": lr.coef_.ravel(),
    "abs_coef": np.abs(lr.coef_.ravel()),
}).sort_values("abs_coef", ascending=False)

print("\nTop 10 features by |coefficient|:")
show(coef_df, n=10)

### <a id="dt"></a> 4.2 — Decision Tree

Recursively splits the feature space into regions, choosing the split that maximizes information gain (or Gini reduction).
Highly interpretable (you can visualize the tree), but prone to **overfitting** — especially with no depth limit.

**Key hyperparameters:** `max_depth`, `min_samples_split`, `min_samples_leaf`.
**Needs scaling:** No — tree splits are based on ordering, not magnitude.

📚 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) · [sklearn Decision Trees Guide](https://scikit-learn.org/stable/modules/tree.html)

In [ ]:
# Shallow tree (controlled)
dt_shallow = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=SEED)
evaluate_model("DecisionTree_d5", dt_shallow, X_train_tr, y_train, X_val_tr, y_val)

# Deep tree (overfitting demo — note train vs val gap)
dt_deep = DecisionTreeClassifier(max_depth=None, min_samples_leaf=1, random_state=SEED)
evaluate_model("DecisionTree_deep", dt_deep, X_train_tr, y_train, X_val_tr, y_val)

# Visualize the shallow tree
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_shallow, max_depth=3, feature_names=feature_names_ohe,
          class_names=["<=50K", ">50K"], filled=True, rounded=True, ax=ax, fontsize=8)
ax.set_title("Decision Tree (max_depth=5, showing top 3 levels)")
plt.tight_layout()
plt.show()

print("\nNotice: DecisionTree_deep has train AUC = 1.0 but lower val AUC — classic overfitting.")

In [ ]:
# ── Decision Tree: complexity vs performance ──
depths = list(range(1, 30))
dt_results = []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    dt.fit(X_train_tr, y_train)
    dt_results.append({
        "max_depth": d,
        "train_auc": roc_auc_score(y_train, dt.predict_proba(X_train_tr)[:, 1]),
        "val_auc": roc_auc_score(y_val, dt.predict_proba(X_val_tr)[:, 1]),
        "n_leaves": dt.get_n_leaves(),
    })
dt_res = pd.DataFrame(dt_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(dt_res["max_depth"], dt_res["train_auc"], "o-", label="Train AUC")
axes[0].plot(dt_res["max_depth"], dt_res["val_auc"], "s-", label="Val AUC")
best_depth = int(dt_res.loc[dt_res["val_auc"].idxmax(), "max_depth"])
axes[0].axvline(x=best_depth, color="red", linestyle="--", alpha=0.5, label=f"Best depth={best_depth}")
axes[0].set_xlabel("max_depth")
axes[0].set_ylabel("AUC")
axes[0].set_title("Decision Tree: Overfitting as Depth Increases")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(dt_res["max_depth"], dt_res["n_leaves"], "o-", color="purple")
axes[1].set_xlabel("max_depth")
axes[1].set_ylabel("Number of Leaves")
axes[1].set_title("Model Complexity (# leaves) vs Depth")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best val AUC at max_depth={best_depth}: {dt_res.loc[dt_res['val_auc'].idxmax(), 'val_auc']:.4f}")
print("Beyond this depth, train AUC keeps rising but val AUC degrades — the model memorizes noise.")

### <a id="rf"></a> 4.3 — Random Forest

An ensemble of decision trees, each trained on a bootstrap sample with random feature subsets.
Reduces variance compared to a single tree. Less prone to overfitting, but less interpretable.

**Key hyperparameters:** `n_estimators` (number of trees), `max_depth`, `max_features` (features per split).
**Needs scaling:** No.

📚 [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) · Breiman (2001), "Random Forests", *Machine Learning*, 45(1), 5–32

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, max_features="sqrt",
    min_samples_leaf=10, random_state=SEED, n_jobs=-1,
)
evaluate_model("RandomForest", rf, X_train_tr, y_train, X_val_tr, y_val)

# MDI feature importance
imp_rf = pd.DataFrame({
    "feature": feature_names_ohe,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
imp_rf.head(15).plot.barh(x="feature", y="importance", ax=ax, legend=False)
ax.set_title("Random Forest — Top 15 Features (MDI)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### <a id="gbm"></a> 4.4 — Gradient Boosting Models

Builds trees **sequentially**, each correcting the errors of the previous ensemble.
The most common family in industry for tabular data (XGBoost, LightGBM, CatBoost).

**Key hyperparameters:** `learning_rate` (shrinkage), `n_estimators` (boosting rounds), `max_depth`.
**Needs scaling:** No. LightGBM can handle categorical features natively (no one-hot encoding needed).

📚 [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) · [XGBoost docs](https://xgboost.readthedocs.io/) · [LightGBM docs](https://lightgbm.readthedocs.io/) · Chen & Guestrin (2016), "XGBoost", *KDD*

In [ ]:
# sklearn GradientBoosting (baseline)
gb_sk = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=4,
    min_samples_leaf=20, random_state=SEED,
)
evaluate_model("GBM_sklearn", gb_sk, X_train_tr, y_train, X_val_tr, y_val)

# XGBoost
xgb_clf = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=5,
    min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, eval_metric="logloss", verbosity=0,
)
evaluate_model("XGBoost", xgb_clf, X_train_tr, y_train, X_val_tr, y_val)

# LightGBM
lgb_clf = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=5,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, verbose=-1,
)
evaluate_model("LightGBM", lgb_clf, X_train_tr, y_train, X_val_tr, y_val)

In [ ]:
# ── Model Zoo Summary ──
RESULTS_DF = pd.DataFrame(RESULTS_ROWS)

display(
    RESULTS_DF[["model", "train_auc", "val_auc", "val_f1", "val_precision",
                "val_recall", "n_params", "fit_time_s"]]
    .sort_values("val_auc", ascending=False)
    .reset_index(drop=True)
    .style.format({
        "train_auc": "{:.4f}", "val_auc": "{:.4f}", "val_f1": "{:.4f}",
        "val_precision": "{:.4f}", "val_recall": "{:.4f}", "fit_time_s": "{:.2f}",
    })
    .bar(subset=["val_auc"], color="#5fba7d")
)

# Flag overfitting: large train-val AUC gap
RESULTS_DF["overfit_gap"] = RESULTS_DF["train_auc"] - RESULTS_DF["val_auc"]
print("\nOverfit gap (train_auc - val_auc):")
print(RESULTS_DF[["model", "overfit_gap"]].sort_values("overfit_gap", ascending=False).to_string(index=False))

> **Do It Yourself — Model Zoo on a New Dataset**
>
> Apply the same modeling workflow to the **Bank Marketing** dataset (45K rows, binary classification).
> The task: predict whether a client subscribes to a term deposit after a phone campaign.
>
> The data is loaded and preprocessed for you below. Your job:
> 1. Train at least one model from each family (LogReg, Decision Tree, Random Forest, GBM).
> 2. Use `evaluate_model()` to collect results (it's the same helper from above).
> 3. Compare: which family works best? Is there overfitting? How does it compare to the Adult dataset?
>
> **Bonus:** Try a variant with different hyperparameters. Can you beat the default configurations?

In [ ]:
# ── Data loading and preprocessing (done for you) ──
from sklearn.datasets import fetch_openml

# Bank Marketing dataset — predict term deposit subscription
bank_data = fetch_openml(data_id=1461, as_frame=True, parser="auto")
bank_df = bank_data.data.copy()
bank_df.columns = [
    "age", "job", "marital", "education", "default", "balance",
    "housing", "loan", "contact", "day", "month", "duration",
    "campaign", "pdays", "previous", "poutcome",
]
bank_target = (bank_data.target == "2").astype(int).values  # 1 = subscribed

print(f"Bank Marketing: {bank_df.shape[0]} rows, {bank_df.shape[1]} features")
print(f"Target rate: {bank_target.mean():.1%} subscribed")

# Identify column types
bank_num_cols = bank_df.select_dtypes(include=["int64", "float64"]).columns.tolist()
bank_cat_cols = bank_df.select_dtypes(include=["category", "object"]).columns.tolist()

# Preprocessing: impute + scale numerics, one-hot encode categoricals
bank_preprocessor_scaled = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), bank_num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="MISSING")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), bank_cat_cols),
])
bank_preprocessor_tree = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), bank_num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="MISSING")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), bank_cat_cols),
])

# Split
X_bank_train, X_bank_val, y_bank_train, y_bank_val = train_test_split(
    bank_df, bank_target, test_size=0.2, random_state=SEED, stratify=bank_target,
)

# Transform
bank_preprocessor_scaled.fit(X_bank_train)
X_bank_train_sc = bank_preprocessor_scaled.transform(X_bank_train)
X_bank_val_sc = bank_preprocessor_scaled.transform(X_bank_val)

bank_preprocessor_tree.fit(X_bank_train)
X_bank_train_tr = bank_preprocessor_tree.transform(X_bank_train)
X_bank_val_tr = bank_preprocessor_tree.transform(X_bank_val)

print(f"\nReady! Train: {X_bank_train_sc.shape}, Val: {X_bank_val_sc.shape}")
print(f"Numeric features: {bank_num_cols}")
print(f"Categorical features: {bank_cat_cols}")

# Fresh registry for this exercise
BANK_REGISTRY = {}
BANK_RESULTS = []

def evaluate_bank_model(name, model, X_tr, y_tr, X_v, y_v):
    """Same as evaluate_model but stores in the bank registry."""
    t0 = time.time()
    model.fit(X_tr, y_tr)
    fit_time = time.time() - t0

    y_prob_tr = model.predict_proba(X_tr)[:, 1]
    y_prob_v = model.predict_proba(X_v)[:, 1]
    y_pred_v = model.predict(X_v)

    row = {
        "model": name,
        "train_auc": roc_auc_score(y_tr, y_prob_tr),
        "val_auc": roc_auc_score(y_v, y_prob_v),
        "val_f1": f1_score(y_v, y_pred_v),
        "fit_time_s": fit_time,
    }
    row["overfit_gap"] = row["train_auc"] - row["val_auc"]
    BANK_RESULTS.append(row)
    BANK_REGISTRY[name] = model

    print(f"[{name}]  Train AUC: {row['train_auc']:.4f}  |  Val AUC: {row['val_auc']:.4f}  |"
          f"  F1: {row['val_f1']:.4f}  |  Gap: {row['overfit_gap']:.4f}  |  Time: {fit_time:.2f}s")
    return model

print("\nUse evaluate_bank_model(name, model, X_tr, y_tr, X_v, y_v)")
print("  → Scaled data: X_bank_train_sc, X_bank_val_sc  (for LogReg)")
print("  → Tree data:   X_bank_train_tr, X_bank_val_tr  (for DT, RF, GBM)")

In [ ]:
# ── YOUR TURN: Train 4 model families on the Bank Marketing dataset ──

# 1. Logistic Regression (use scaled data)
# TODO: evaluate_bank_model("LR", ..., X_bank_train_sc, y_bank_train, X_bank_val_sc, y_bank_val)


# 2. Decision Tree (use tree data)
# TODO: evaluate_bank_model("DT", ..., X_bank_train_tr, y_bank_train, X_bank_val_tr, y_bank_val)


# 3. Random Forest (use tree data)
# TODO: evaluate_bank_model("RF", ..., X_bank_train_tr, y_bank_train, X_bank_val_tr, y_bank_val)


# 4. Gradient Boosting (use tree data)
# TODO: evaluate_bank_model("GBM", ..., X_bank_train_tr, y_bank_train, X_bank_val_tr, y_bank_val)


# 5. Compare results
# pd.DataFrame(BANK_RESULTS).sort_values("val_auc", ascending=False)

### Metrics in Practice

Now that we have trained models, let's see these metrics in action.
We look at the confusion matrix, threshold sensitivity, and ROC/PR curves for the best model.

In [ ]:
# ── Confusion Matrix and Threshold-Dependent Metrics ──
best_name = RESULTS_DF.sort_values("val_auc", ascending=False).iloc[0]["model"]
best_entry = MODEL_REGISTRY[best_name]
best_model = best_entry["model"]
y_prob_best = best_entry["y_prob_val"]
y_pred_best = best_entry["y_pred_val"]

print(f"Using model: {best_name}\n")
print(classification_report(y_val, y_pred_best, target_names=["<=50K", ">50K"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_best, display_labels=["<=50K", ">50K"], ax=ax, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

In [ ]:
# ── Threshold Sensitivity ──
thresholds = np.arange(0.05, 0.96, 0.05)
threshold_metrics = []
for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    if y_pred_t.sum() == 0 or y_pred_t.sum() == len(y_pred_t):
        continue
    threshold_metrics.append({
        "threshold": t,
        "precision": precision_score(y_val, y_pred_t, zero_division=0),
        "recall": recall_score(y_val, y_pred_t),
        "f1": f1_score(y_val, y_pred_t),
        "accuracy": accuracy_score(y_val, y_pred_t),
    })
tm_df = pd.DataFrame(threshold_metrics)

fig, ax = plt.subplots(figsize=(12, 5))
for col in ["precision", "recall", "f1", "accuracy"]:
    ax.plot(tm_df["threshold"], tm_df[col], label=col, marker=".")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Metric Value")
ax.set_title(f"Threshold vs Metrics — {best_name}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The default threshold of 0.5 is not always optimal.")
print("In practice, pick the threshold based on business constraints (cost of FP vs FN).")

In [ ]:
# ── ROC and Precision-Recall Curves (multi-model overlay) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
for name, entry in MODEL_REGISTRY.items():
    prob = entry["y_prob_val"]
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc_val = roc_auc_score(y_val, prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

# Precision-Recall curves
for name, entry in MODEL_REGISTRY.items():
    prob = entry["y_prob_val"]
    prec, rec, _ = precision_recall_curve(y_val, prob)
    ap = average_precision_score(y_val, prob)
    axes[1].plot(rec, prec, label=f"{name} (AP={ap:.3f})")
baseline_rate = y_val.mean()
axes[1].axhline(y=baseline_rate, color="k", linestyle="--", alpha=0.4, label=f"Baseline ({baseline_rate:.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves")
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("AUC-ROC summarizes overall discriminative ability.")
print("AUC-PR is more informative when the positive class is rare.")

> **Do It Yourself**
>
> Pick the best model under two constraints:
> - **(a)** Maximize recall at precision ≥ 0.7
> - **(b)** Minimize false positives (maximize precision at recall ≥ 0.5)
>
> What threshold would you use in each scenario?

## <a id="overfitting"></a> Section 5 — The Overfitting Problem

Overfitting happens when a model memorizes the training data instead of learning generalizable patterns.
The result: excellent training performance, poor validation/test performance.

We already saw this with Decision Trees (§3.2). Now we look at GBM and add **learning curves**.

- If train and val both improve with more data → you're **underfitting** (model needs more capacity or data).
- If train is high but val plateaus → you're **overfitting** (model is too complex for the data).

📚 [sklearn Learning Curves](https://scikit-learn.org/stable/modules/learning_curve.html) · Hastie, Tibshirani & Friedman, *Elements of Statistical Learning*, Ch. 7

In [ ]:
# ── GBM: n_estimators vs performance ──
n_est_range = [10, 25, 50, 100, 200, 300, 500, 800]
gb_results = []
for n in tqdm(n_est_range):
    gb = GradientBoostingClassifier(
        n_estimators=n, learning_rate=0.1, max_depth=4,
        min_samples_leaf=20, random_state=SEED,
    )
    gb.fit(X_train_tr, y_train)
    gb_results.append({
        "n_estimators": n,
        "train_auc": roc_auc_score(y_train, gb.predict_proba(X_train_tr)[:, 1]),
        "val_auc": roc_auc_score(y_val, gb.predict_proba(X_val_tr)[:, 1]),
    })
gb_res = pd.DataFrame(gb_results)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gb_res["n_estimators"], gb_res["train_auc"], "o-", label="Train AUC")
ax.plot(gb_res["n_estimators"], gb_res["val_auc"], "s-", label="Val AUC")
ax.set_xlabel("n_estimators")
ax.set_ylabel("AUC")
ax.set_title("Gradient Boosting: More Trees = More Overfitting Risk")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("With boosting, adding more trees always improves train performance.")
print("But val performance plateaus and can degrade — early stopping is the standard remedy.")

## <a id="tuning"></a> Section 6 — Hyperparameter Tuning

Manual hyperparameter search is tedious and error-prone.
sklearn provides `GridSearchCV` to automate this inside a cross-validation loop.

**Critical**: when tuning is done inside a `Pipeline`, preprocessing is re-fitted on each CV fold's training data, **preventing data leakage**.
This is the single most important pattern in this notebook — it connects preprocessing (Day 1) with model selection (Day 2).

For larger search spaces, `RandomizedSearchCV` samples random configurations instead of exhaustive search (1-sentence summary — see Advanced notebook for Bayesian optimization with Optuna).

📚 [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) · Bergstra & Bengio (2012), "Random Search for Hyper-Parameter Optimization", *JMLR*

In [ ]:
# ── Preprocessing leakage demo ──
# Show WHY Pipeline + GridSearchCV matters: leaking preprocessing gives optimistic results.

cat_cols_demo = [c for c in ["occupation", "workclass", "marital_status", "relationship", "native_country"]
                 if c in train_df.columns]

# --- Correct: OHE fitted on train only ---
ohe_correct = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe_correct.fit(train_df[cat_cols_demo].fillna("MISSING").astype(str))
X_train_ohe_c = ohe_correct.transform(train_df[cat_cols_demo].fillna("MISSING").astype(str))
X_test_ohe_c = ohe_correct.transform(test_df[cat_cols_demo].fillna("MISSING").astype(str))

# --- Leaky: OHE fitted on full dataset ---
full_cat = pd.concat([train_df[cat_cols_demo], val_df[cat_cols_demo], test_df[cat_cols_demo]]).fillna("MISSING").astype(str)
ohe_leaky = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe_leaky.fit(full_cat)
X_train_ohe_l = ohe_leaky.transform(train_df[cat_cols_demo].fillna("MISSING").astype(str))
X_test_ohe_l = ohe_leaky.transform(test_df[cat_cols_demo].fillna("MISSING").astype(str))

# Build numeric features
num_cols_demo = [c for c in numeric_cols if c in X_train_raw.columns]
imp_demo = SimpleImputer(strategy="median")
X_train_num_demo = imp_demo.fit_transform(X_train_raw[num_cols_demo])
X_test_num_demo = imp_demo.transform(X_test_raw[num_cols_demo])

X_train_correct = np.hstack([X_train_num_demo, X_train_ohe_c])
X_test_correct = np.hstack([X_test_num_demo, X_test_ohe_c])
X_train_leaky = np.hstack([X_train_num_demo, X_train_ohe_l])
X_test_leaky = np.hstack([X_test_num_demo, X_test_ohe_l])

# Train same model both ways
for label, X_tr, X_te in [("Correct (train-only OHE)", X_train_correct, X_test_correct),
                           ("Leaky (full-data OHE)", X_train_leaky, X_test_leaky)]:
    m = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED)
    m.fit(X_tr, y_train)
    test_auc = roc_auc_score(y_test, m.predict_proba(X_te)[:, 1])
    print(f"  {label:30s} → Test AUC: {test_auc:.4f}  (features: {X_tr.shape[1]})")

print("\nThe leaky pipeline has access to categories it shouldn't know about at training time.")
print("Pipeline + GridSearchCV prevents this automatically.")

### Why tuning matters: same model, wildly different results

Before we automate tuning with GridSearchCV, let's see **how much** hyperparameters matter.
Below we train the same Random Forest with 5 different configurations — from terrible to well-tuned.
The AUC spread is dramatic: the "same algorithm" can look useless or excellent depending on its settings.

In [ ]:
# ── Hyperparameter sensitivity: same algorithm, very different outcomes ──
rf_configs = {
    "RF: stumps (depth=1, 10 trees)": dict(
        n_estimators=10, max_depth=1, random_state=SEED, n_jobs=-1,
    ),
    "RF: shallow (depth=3, 10 trees)": dict(
        n_estimators=10, max_depth=3, random_state=SEED, n_jobs=-1,
    ),
    "RF: no limit (depth=None, 10 trees)": dict(
        n_estimators=10, max_depth=None, random_state=SEED, n_jobs=-1,
    ),
    "RF: decent (depth=10, 100 trees)": dict(
        n_estimators=100, max_depth=10, min_samples_leaf=10, random_state=SEED, n_jobs=-1,
    ),
    "RF: well-tuned (depth=15, 300 trees)": dict(
        n_estimators=300, max_depth=15, min_samples_leaf=5, max_features="sqrt",
        random_state=SEED, n_jobs=-1,
    ),
}

hp_results = []
for name, params in rf_configs.items():
    rf_hp = RandomForestClassifier(**params)
    rf_hp.fit(X_train_tr, y_train)
    tr_auc = roc_auc_score(y_train, rf_hp.predict_proba(X_train_tr)[:, 1])
    v_auc = roc_auc_score(y_val, rf_hp.predict_proba(X_val_tr)[:, 1])
    hp_results.append({"config": name, "train_auc": tr_auc, "val_auc": v_auc, "gap": tr_auc - v_auc})

hp_df = pd.DataFrame(hp_results)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
x_pos = range(len(hp_df))
bars_train = ax.bar([p - 0.15 for p in x_pos], hp_df["train_auc"], width=0.3,
                     label="Train AUC", color="steelblue", alpha=0.7)
bars_val = ax.bar([p + 0.15 for p in x_pos], hp_df["val_auc"], width=0.3,
                   label="Val AUC", color="darkorange", alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace("RF: ", "") for c in hp_df["config"]], rotation=15, ha="right")
ax.set_ylabel("AUC")
ax.set_title("Same Algorithm (Random Forest), Different Hyperparameters → Different Results")
ax.legend()
ax.set_ylim(0.5, 1.02)
ax.grid(True, alpha=0.3, axis="y")

# Annotate val AUC on bars
for bar, val in zip(bars_val, hp_df["val_auc"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Val AUC range: {hp_df['val_auc'].min():.4f} → {hp_df['val_auc'].max():.4f}"
      f"  (spread = {hp_df['val_auc'].max() - hp_df['val_auc'].min():.4f})")
print()
print("Key observations:")
print("• Too shallow (depth=1): underfits — can't capture patterns → low AUC")
print("• Too few trees (10): high variance, unstable estimates")
print("• No depth limit + few trees: overfits — train AUC >> val AUC")
print("• Well-tuned: balanced depth, enough trees, controlled leaf size → best val AUC")
print()
print("This is why we need systematic hyperparameter search — manual guessing is unreliable.")

In [ ]:
# ── GridSearchCV with GBM Pipeline ──
pipe_gbm = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("clf", GradientBoostingClassifier(random_state=SEED)),
])

param_grid_gbm = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [3, 4, 5],
    "clf__learning_rate": [0.05, 0.1],
}

gs_gbm = GridSearchCV(
    pipe_gbm,
    param_grid_gbm,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring="roc_auc",
    n_jobs=-1,
    refit=True,
)
gs_gbm.fit(X_train_raw, y_train)

print(f"Best params: {gs_gbm.best_params_}")
print(f"Best CV AUC: {gs_gbm.best_score_:.4f}")

val_auc_gbm = roc_auc_score(y_val, gs_gbm.predict_proba(X_val_raw)[:, 1])
print(f"Val AUC (hold-out): {val_auc_gbm:.4f}")

# Show top-5 configurations
cv_results = pd.DataFrame(gs_gbm.cv_results_).sort_values("rank_test_score")
show(cv_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]], n=5)

> **Do It Yourself**
>
> Build a Pipeline for RandomForest. Tune `n_estimators` and `max_depth` with `GridSearchCV`.
> Compare the best CV score to the hold-out val score — are they close?

## <a id="diagnostics"></a> Section 7 — Model Diagnostics

Going beyond aggregate metrics: **understanding how and where the model succeeds or fails**.

We cover two core diagnostics:
- **Permutation importance** — which features matter most?
- **Model comparison** — how do models stack up overall?

For SHAP, calibration deep-dives, and contrast model error analysis, see the **Advanced** notebook.

### <a id="importance"></a> 7.1 — Permutation Importance

**Permutation importance** (model-agnostic): shuffle one feature at a time, measure how much the metric drops.
Unlike tree-based MDI importance, this works for any model and is computed on held-out data.

📚 [Permutation Importance](https://scikit-learn.org/stable/modules/permutation_importance.html)

In [ ]:
# ── Permutation Importance ──

# Use a tree-based model for diagnostics (fast + good performance)
diag_model_name = "RandomForest"
if diag_model_name not in MODEL_REGISTRY:
    diag_model_name = list(MODEL_REGISTRY.keys())[-1]
diag_model = MODEL_REGISTRY[diag_model_name]["model"]

perm_imp = permutation_importance(
    diag_model, X_val_tr, y_val,
    n_repeats=10, random_state=SEED, scoring="roc_auc", n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": feature_names_ohe,
    "importance_mean": perm_imp.importances_mean,
    "importance_std": perm_imp.importances_std,
}).sort_values("importance_mean", ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
top_perm = perm_df.head(15)
ax.barh(top_perm["feature"], top_perm["importance_mean"],
        xerr=top_perm["importance_std"], color="steelblue")
ax.set_xlabel("Mean AUC decrease")
ax.set_title(f"Permutation Importance ({diag_model_name}, val set)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### <a id="comparison"></a> 7.3 — Model Comparison

In [ ]:
# ── Final comparison table ──
final_df = RESULTS_DF.copy()
final_df["overfit_gap"] = final_df["train_auc"] - final_df["val_auc"]

display(
    final_df[["model", "train_auc", "val_auc", "overfit_gap", "val_f1",
              "val_precision", "val_recall", "n_params", "fit_time_s"]]
    .sort_values("val_auc", ascending=False)
    .reset_index(drop=True)
    .style.format({
        "train_auc": "{:.4f}", "val_auc": "{:.4f}", "overfit_gap": "{:.4f}",
        "val_f1": "{:.4f}", "val_precision": "{:.4f}", "val_recall": "{:.4f}",
        "fit_time_s": "{:.2f}",
    })
    .background_gradient(subset=["val_auc"], cmap="Greens")
    .background_gradient(subset=["overfit_gap"], cmap="Reds")
)

In [ ]:
# ── Visual comparison ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = final_df.sort_values("val_auc", ascending=True)
colors = ["#e74c3c" if g > 0.05 else "#5fba7d" for g in plot_df["overfit_gap"]]
axes[0].barh(plot_df["model"], plot_df["val_auc"], color=colors)
axes[0].set_xlabel("Val AUC")
axes[0].set_title("Validation AUC by Model (red = overfit gap > 0.05)")
axes[0].set_xlim(0.5, 1.0)

axes[1].scatter(plot_df["fit_time_s"], plot_df["val_auc"], s=80, zorder=3)
for _, row in plot_df.iterrows():
    axes[1].annotate(row["model"], (row["fit_time_s"], row["val_auc"]),
                     fontsize=7, ha="left", va="bottom")
axes[1].set_xlabel("Fit Time (seconds)")
axes[1].set_ylabel("Val AUC")
axes[1].set_title("Performance vs Training Cost")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key takeaway: there is no single best model.")
print("The choice depends on: performance, interpretability, training cost, calibration needs, and deployment constraints.")

> **Do It Yourself**
>
> Look at the permutation importance. Are the top features the ones you expected from Day 1's EDA?
> If not, what might explain the difference between what looks important in EDA vs what matters for prediction?

## <a id="checklist"></a> Section 8 — End-of-Notebook Checklist

> **Do It Yourself**
>
> Fill this checklist for your best model before considering it "ready":

> - [ ] Splitting strategy matches data structure (group/time/stratified as needed).
> - [ ] Metrics chosen are appropriate for the problem and business context.
> - [ ] Overfitting checked (train vs val gap).
> - [ ] Cross-validation used for hyperparameter selection.
> - [ ] Pipeline ensures no preprocessing leakage inside CV.
> - [ ] Permutation importance inspected.
> - [ ] Calibration assessed.
> - [ ] Results are reproducible (fixed seeds, logged hyperparameters).

In [ ]:
reference_checklist = {
    "dataset": "adult_income_issues.csv",
    "split_strategy": "Stratified + entity-aware (person_id), using provided split column",
    "metrics_chosen": "AUC-ROC (primary), F1, Precision, Recall",
    "overfitting_checked": True,
    "cv_for_tuning": True,
    "pipeline_no_leakage": True,
    "permutation_importance": True,
    "calibration_assessed": True,
    "reproducible": True,
}

pd.DataFrame(reference_checklist, index=[0]).T.rename(columns={0: "Status"})

---

## References

### Models
- scikit-learn, [Supervised Learning Guide](https://scikit-learn.org/stable/supervised_learning.html)
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32.
- Friedman, J. H. (2001). Greedy Function Approximation: A Gradient Boosting Machine. *Annals of Statistics*, 29(5), 1189–1232.
- Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD*.
- Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *NeurIPS*.

### Evaluation & Model Selection
- scikit-learn, [Model Evaluation Guide](https://scikit-learn.org/stable/modules/model_evaluation.html)
- scikit-learn, [Cross-validation Guide](https://scikit-learn.org/stable/modules/cross_validation.html)
- Bergstra, J. & Bengio, Y. (2012). Random Search for Hyper-Parameter Optimization. *JMLR*, 13, 281–305.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. Ch. 7.

### Diagnostics
- Platt, J. (1999). Probabilistic Outputs for Support Vector Machines.

## <a id="acceptance"></a> Acceptance Checks

These checks validate the Core Day 2 modeling workflow end-to-end.

In [ ]:
# 1) Data integrity
assert X_train_sc.shape[0] == len(y_train)
assert X_val_sc.shape[0] == len(y_val)
assert np.isfinite(X_train_sc).all()
assert np.isfinite(X_train_tr).all()

# 2) Model registry: at least 4 model families trained
assert len(MODEL_REGISTRY) >= 4, f"Expected >= 4 models, got {len(MODEL_REGISTRY)}"

# 3) Metric consistency: all models evaluated with same metric set
assert all(col in RESULTS_DF.columns for col in ["train_auc", "val_auc", "val_f1"])

# 4) Overfitting check: at least one model shows train >> val gap
assert (RESULTS_DF["train_auc"] - RESULTS_DF["val_auc"]).max() > 0.01, "Expected at least one model with overfit gap"

# 5) Diagnostic outputs produced
assert len(perm_df) > 0, "Permutation importance not computed"

# 6) Reproducibility
assert SEED == 42

print("All acceptance checks passed.")
print("Core Day 2 notebook is ready.")